In [1]:
import sys, time, gc
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, joblib

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.features import to_matrix, ClippedScaler
from src.evaluation import evaluate
from src.pipeline import DATA, RESULTS, score_split

SEEDS = [11, 22, 33, 44, 55]
d = joblib.load(RESULTS / "scaler_v2.joblib")
cols, scaler = d["cols"], d["scaler"]
(RESULTS / "scores").mkdir(exist_ok=True)

for seed in SEEDS:
    t0 = time.time()
    torch.manual_seed(seed); np.random.seed(seed)

    tr = pd.read_parquet(DATA / "train.parquet", columns=cols).sample(1_000_000, random_state=seed)
    x_train = scaler.transform(to_matrix(tr, cols)).astype(np.float32)
    va = pd.read_parquet(DATA / "val.parquet", columns=cols).sample(100_000, random_state=seed)
    x_val = scaler.transform(to_matrix(va, cols)).astype(np.float32)
    del tr, va; gc.collect()

    dim = x_train.shape[1]
    model = nn.Sequential(
        nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 12), nn.ReLU(),
        nn.Linear(12, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, dim))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    Xtr, Xva = torch.from_numpy(x_train), torch.from_numpy(x_val)

    best, best_state, bad = float("inf"), None, 0
    for epoch in range(30):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 1024):
            xb = Xtr[perm[i:i + 1024]]
            opt.zero_grad(); loss_fn(model(xb), xb).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(Xva), Xva).item()
        if v < best - 1e-4:
            best, bad = v, 0
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3:
                break
    model.load_state_dict(best_state)

    def ae_score(x):
        model.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(x), 100_000):
                xb = torch.from_numpy(x[i:i + 100_000])
                out.append(((model(xb) - xb) ** 2).mean(dim=1).numpy())
        return np.concatenate(out)

    parts = [score_split(sp, cols, scaler, ae_score, cap=(100_000 if sp == "test" else None))
             for sp in ["val", "tune", "test"]]
    pd.concat(parts).to_parquet(RESULTS / "scores" / f"ms_m3v2_seed{seed}.parquet")
    print(f"seed {seed}: epochs {epoch + 1}, best val loss {best:.5f}, {time.time() - t0:.0f}s")
    del x_train, x_val, Xtr, Xva, model; gc.collect()

seed 11: epochs 23, best val loss 0.00263, 122s
seed 22: epochs 23, best val loss 0.00314, 116s
seed 33: epochs 17, best val loss 0.00438, 90s
seed 44: epochs 25, best val loss 0.00256, 116s
seed 55: epochs 26, best val loss 0.00302, 116s


In [2]:
from src.tcn import TCNMAE, random_mask, masked_loss, window_scores

def load(split):
    z = np.load(DATA / "windows" / f"{split}.npz", allow_pickle=True)
    r = {k: z[k] for k in z.files}
    r["attack_type"] = r["attack_type"].astype(str)
    return r

tr, va, tu, te = load("train"), load("val"), load("tune"), load("test")
D = tr["X"].shape[2]
Xtr, Vtr = torch.from_numpy(tr["X"]), torch.from_numpy(tr["mask"])
Xva, Vva = torch.from_numpy(va["X"]), torch.from_numpy(va["mask"])

def to_df(split, dd):
    return pd.DataFrame({"id": dd["win_ids"], "score": window_scores(model, dd["X"], dd["mask"]),
                         "label": dd["label"].astype(int), "attack_type": dd["attack_type"], "split": split})

for seed in SEEDS:
    t0 = time.time()
    torch.manual_seed(seed); np.random.seed(seed)
    model = TCNMAE(D)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    Mva = random_mask(Vva)

    best, best_state, bad = float("inf"), None, 0
    for epoch in range(25):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 512):
            idx = perm[i:i + 512]
            x, v = Xtr[idx], Vtr[idx]
            opt.zero_grad(); masked_loss(model, x, v, random_mask(v)).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = np.mean([masked_loss(model, Xva[i:i + 2048], Vva[i:i + 2048], Mva[i:i + 2048]).item()
                          for i in range(0, len(Xva), 2048)])
        if vl < best - 1e-4:
            best, bad = vl, 0
            best_state = {k: t.clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 3:
                break
    model.load_state_dict(best_state)
    pd.concat([to_df("val", va), to_df("tune", tu), to_df("test", te)]).to_parquet(
        RESULTS / "scores" / f"ms_m4_seed{seed}.parquet")
    print(f"seed {seed}: epochs {epoch + 1}, best val loss {best:.4f}, {time.time() - t0:.0f}s")

seed 11: epochs 19, best val loss 0.4036, 303s
seed 22: epochs 11, best val loss 0.4086, 191s
seed 33: epochs 16, best val loss 0.4058, 264s
seed 44: epochs 16, best val loss 0.4052, 261s
seed 55: epochs 20, best val loss 0.4028, 316s


In [4]:
def summarize(prefix):
    rows = []
    for s in SEEDS:
        df = pd.read_parquet(RESULTS / "scores" / f"{prefix}_seed{s}.parquet")
        val = df[df["split"] == "val"]
        for sp in ["tune", "test"]:
            d_ = df[df["split"] == sp]
            r, o = evaluate(d_, val_df=val), evaluate(d_)
            rows.append({"seed": s, "split": sp, "auroc": r["auroc"],
                         "macro_tpr@5%": r["macro_tpr@fpr0.05"], "real_fpr@5%": r["real_fpr@0.05"],
                         "oracle_macro_tpr@5%": o["macro_tpr@fpr0.05"]})
    t = pd.DataFrame(rows)
    t.to_csv(RESULTS / f"{prefix}_per_seed.csv", index=False)
    return t.drop(columns="seed").groupby("split").agg(["mean", "std"]).round(3)

print("M3 v2 (flow level)"); print(summarize("ms_m3v2"))
print("\nM4 TCN (window level)"); print(summarize("ms_m4")) 

M3 v2 (flow level)
       auroc        macro_tpr@5%        real_fpr@5%         \
        mean    std         mean    std        mean    std   
split                                                        
test   0.415  0.022        0.948  0.016       0.831  0.033   
tune   0.952  0.016        0.894  0.084       0.090  0.005   

      oracle_macro_tpr@5%         
                     mean    std  
split                             
test                0.080  0.060  
tune                0.496  0.118  

M4 TCN (window level)
       auroc        macro_tpr@5%        real_fpr@5%         \
        mean    std         mean    std        mean    std   
split                                                        
test   0.481  0.005        0.661  0.014       0.696  0.015   
tune   0.568  0.007        0.590  0.004       0.106  0.001   

      oracle_macro_tpr@5%         
                     mean    std  
split                             
test                0.036  0.020  
tune                0